[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Your First Class


## What you will be able to do

Write a class of your own: define it, give it an `__init__` that stores what each object needs,
create several objects from it, and read and change what each one is carrying.


## The idea

### The problem

The **Why Classes** notebook showed a `Station` class and said outright that the mechanics were
being skipped. Here they are.

There is a particular way this goes wrong for people. The shape of a class is easy to copy: type
`class`, type `def __init__(self, ...)`, write `self.name = name` a few times, and it works. What
is much harder is answering three questions about the thing you just copied.

**Where does `self` come from?** You never pass it. `Station("Tromso", readings, "C")` supplies
three values and `__init__` declares four parameters, and Python does not complain.

**Who calls `__init__`?** You call `Station(...)`. Nothing in your code mentions `__init__`, and
yet it runs.

**Why write every name twice?** `self.name = name` looks like it is doing nothing, and the fact
that both halves are usually spelled the same way makes it worse.

Copying a shape you cannot explain works until the first error message, which will mention `self`,
and then there is nothing to reason from. All three questions have the same answer, and it is
worth getting properly rather than by pattern-matching.

### What happens when you create an object

> A `class` statement creates a new **type**. Calling that type, `Station(...)`, does two things:
> it makes a new empty object, and then it calls `__init__` with that object as the **first**
> argument, followed by whatever else was in the call. `__init__` stores what the object needs on
> it. The name `self` is simply the name that first parameter is conventionally given.

### Why it works that way

Read the definition again with the three questions in mind, because it answers all of them.

`self` comes from Python, not from you. It is the new object, passed in as the first argument.
That is why `Station("Tromso", readings, "C")` passes three values to a function declaring four
parameters: Python supplies the first one.

`__init__` is called by Python, immediately after the empty object exists. The name is fixed. Call
your method `setup` and nothing will run it.

And `self.name = name` writes two different things. On the left, `self.name` is an attribute on the
new object, which did not exist a moment ago. On the right, `name` is the parameter, a local
variable that disappears when `__init__` returns. The line copies the argument onto the object so
that it outlives the call. They are usually spelled the same because that reads best, not because
they are related.

One consequence is worth stating early: `__init__` does not create the object and does not return
it. Python already made it. `__init__` only fills it in, which is why returning anything from it is
an error.

Another is that each object gets its own attributes. Two stations created from one class share no
data at all, which is exactly what makes them useful as separate values.

### Where you will meet this

Every class you have already used. `Path("data.csv")` runs `Path.__init__`. So does
`ZipFile(archive)`, and every exception you have ever caught. When you meet an unfamiliar class in
somebody's code, its `__init__` is the fastest way to learn what the thing needs to exist, because
its parameters are exactly that list.

### What this notebook covers

A class with nothing in it, then attributes attached by hand, then `__init__` doing it properly.
What `self` is, shown rather than asserted. Defaults, and the fact that every object carries its
own data. Methods you call yourself are the **Methods** notebook; the only one here is `__init__`,
which Python calls for you.

### A first look

Four lines, and two objects made from them. There is nothing to run yet: read it, and read the
output underneath it.

```python
class Station:
    def __init__(self, name, unit):
        self.name = name
        self.unit = unit


north = Station("Tromso", "C")
south = Station("Malaga", "F")
print(north.name, north.unit)
print(south.name, south.unit)
```

```
Tromso C
Malaga F
```

Two objects, four attributes, and no interference between them.


## Setup

One import, used once near the end.

- `mean` averages a list of numbers, for the single method this notebook defines

Everything else here is built as it goes, so this cell is short.

**Run this cell before the rest of the notebook.**


In [1]:
from statistics import mean

print("ready")


ready


## Worked examples

### A class with nothing in it

`class` followed by a name and a body is enough. `pass` is there because Python requires a body and
there is nothing to put in it yet.

Calling `Bare()` makes an object. Calling it twice makes two.


In [2]:
class Bare:
    pass


north = Bare()
south = Bare()

print("type of north:", type(north))
print("north is south:", north is south)
print("what north carries:", vars(north))


type of north: <class '__main__.Bare'>
north is south: False
what north carries: {}


`vars(object)` returns the attributes an object is holding, as a dictionary. Both objects are
holding nothing, because nothing has put anything on them.

`north is south` is `False`, so these are two separate objects and not two names for one. That is
the **Lists** rule again, and it is what makes each call to `Bare()` worth making.

### Attributes attached by hand

Attributes can be assigned from outside, with ordinary assignment. This is real Python and it
works.


In [3]:
north.name = "Tromso"
north.unit = "C"

print("north.name:      ", north.name)
print("north carries:   ", vars(north))
print("south carries:   ", vars(south))


north.name:       Tromso
north carries:    {'name': 'Tromso', 'unit': 'C'}
south carries:    {}


`north` now has two attributes and `south` still has none, from the same class.

That is the problem with doing it this way. Nothing about `Bare` says a station has a name. Every
place in the program that creates one has to remember to attach the same attributes, spelled the
same way, and a single missed line produces an object that looks fine until something reads the
attribute that is not there.

What is needed is a way to run that setup automatically, every time, as part of creating the
object.

### `__init__` is that setup

Give the class a method called `__init__` and Python calls it for you whenever an object is
created. To show that plainly, this one prints before it stores anything.


In [4]:
class Loud:
    def __init__(self, name):
        print("  __init__ is running, and name is", name)
        self.name = name


print("about to create an object")
made = Loud("Tromso")
print("created, and it carries", vars(made))


about to create an object
  __init__ is running, and name is Tromso
created, and it carries {'name': 'Tromso'}


Nothing in that cell says `__init__`. The line `Loud("Tromso")` ran it.

The name is fixed and it is not a convention you can vary. A method called `setup` or `init` is
just a method, and Python will not call it for you.

### What `self` actually is

`self` is the first parameter of `__init__`, and Python passes the new object into it. It is not a
keyword, and the name is not enforced. This class proves it by using a different one.


In [5]:
class Odd:
    def __init__(whatever, name):
        whatever.name = name


print("it works:", Odd("Bodo").name)


it works: Bodo


This runs, and you should never write it. The point is only that `self` is an ordinary parameter
name. Everybody writes `self`, every editor and error message assumes it, and a class that uses
another name will confuse every reader including you.

Now the two steps, separated. `Station(...)` makes an empty object and then calls `__init__` on it,
and those two things can be done by hand.

You will not write this in real code. It is here because it is the only way to watch the object
exist before `__init__` has touched it.


In [6]:
class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit


blank = Station.__new__(Station)
print("step 1, a new empty object:", vars(blank))

Station.__init__(blank, "Tromso", [-4.1, -2.6, -3.8])
print("step 2, after __init__ ran: ", vars(blank))

print()
print("the normal way, both steps at once:")
print(" ", vars(Station("Tromso", [-4.1, -2.6, -3.8])))


step 1, a new empty object: {}
step 2, after __init__ ran:  {'name': 'Tromso', 'readings': [-4.1, -2.6, -3.8], 'unit': 'C'}

the normal way, both steps at once:
  {'name': 'Tromso', 'readings': [-4.1, -2.6, -3.8], 'unit': 'C'}


`Station.__init__(blank, "Tromso", [...])` is the call Python makes on your behalf, written out.
`blank` is the object, and inside `__init__` that object is called `self`.

That is the whole of it. `self` is the object the method was called on, passed in as the first
argument.

### `self.name = name` copies the argument onto the object

The left side and the right side are different things that happen to share a spelling. This version
spells them differently so the two halves are visible.


In [7]:
class Spelled:
    def __init__(self, given_name, given_readings):
        self.name = given_name
        self.readings = given_readings


spelled = Spelled("Bodo", [-2.6, -1.9])

print("attributes stored:", vars(spelled))
print("read one back:    ", spelled.name)


attributes stored: {'name': 'Bodo', 'readings': [-2.6, -1.9]}
read one back:     Bodo


`given_name` is a local variable inside `__init__`, and it is gone the moment `__init__` returns.
`self.name` is on the object, and it lasts as long as the object does. The assignment is what
carries the value from the first to the second.

Writing `self.name = name` instead is better style, because the parameter name is also what
callers see when they read the signature. Just do not read it as a line that does nothing.

### Every object carries its own data

Two stations from one class. Changing one changes nothing about the other.


In [8]:
north = Station("Tromso", [-4.1, -2.6, -3.8])
south = Station("Malaga", [66.0, 67.1], "F")

south.readings.append(68.2)
south.unit = "F"

print("south:", vars(south))
print("north:", vars(north))


south: {'name': 'Malaga', 'readings': [66.0, 67.1, 68.2], 'unit': 'F'}
north: {'name': 'Tromso', 'readings': [-4.1, -2.6, -3.8], 'unit': 'C'}


This is what the **Why Classes** notebook was reaching for. `north.unit` cannot be reached from
`south`, so the mismatch that the loose-variable version allowed has nowhere to happen.

### Defaults, for values that usually do not change

`__init__` is a function, so its parameters can have defaults, exactly as the **Functions** notebook
described. `unit="C"` above is one already. Here is a second.


In [9]:
class Station:
    def __init__(self, name, readings, unit="C", calibrated=True):
        self.name = name
        self.readings = readings
        self.unit = unit
        self.calibrated = calibrated

    def average(self):
        return f"{mean(self.readings):.1f} {self.unit}"


north = Station("Tromso", [-4.1, -2.6, -3.8])
south = Station("Malaga", [66.0, 67.1, 68.2], "F", calibrated=False)

print(north.name, north.average(), "calibrated:", north.calibrated)
print(south.name, south.average(), "calibrated:", south.calibrated)


Tromso -3.5 C calibrated: True
Malaga 67.1 F calibrated: False


`average` is a method you call yourself, and it is the only one in this notebook. Note that it takes
`self` and nothing else, and that it reaches its two values through `self` rather than being handed
them. The **Methods** notebook is about writing these.

A default makes a parameter optional without making the attribute optional. Every station has a
`calibrated` attribute; most of them just did not have to say so.

### Attributes can still be added later, and usually should not be

Assignment from outside works on any object, whether or not `__init__` set the name up.


In [10]:
north.owner = "MET Norway"

print("north:", vars(north))
print("south:", vars(south))
print("does south have an owner:", hasattr(south, "owner"))


north: {'name': 'Tromso', 'readings': [-4.1, -2.6, -3.8], 'unit': 'C', 'calibrated': True, 'owner': 'MET Norway'}
south: {'name': 'Malaga', 'readings': [66.0, 67.1, 68.2], 'unit': 'F', 'calibrated': False}
does south have an owner: False


Now two objects of the same class carry different attributes, and any code reading `station.owner`
works for one and raises for the other.

This is worth knowing about, because it explains errors you will see, and because occasionally it
is what you want. As a habit it removes the main thing `__init__` is for: a guarantee that every
object of this class carries the same set of attributes. If a station may or may not have an owner,
say so in `__init__` with a default of `None`.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/02-your-first-class-solutions.ipynb).

**1.** Write a `Book` class whose `__init__` takes and stores `title`, `author` and `pages`. Create
two books and print `vars` of each.


In [11]:
# your code here


**2.** Add a fourth parameter, `read_so_far`, defaulting to `0`. Create one book without it and one
with it, and print `vars` of each.


In [12]:
# your code here


**3.** This class stores nothing, because both assignments go to local variables. Fix it, then
create one and print the attribute.

```python
class Reading:
    def __init__(self, station, celsius):
        station = station
        celsius = celsius
```


In [13]:
# your code here


**4.** Create two books, change `read_so_far` on one of them, and print that attribute from both,
to show that objects do not share.


In [14]:
# your code here


**5.** Write a `Tagged` class whose `__init__` takes `title` and `tags=[]`. Create two without
tags, append to one, and print both. Then write the version that does not have this problem, and
show the difference.


In [15]:
# your code here


**6.** Give `Book` a `summary()` method returning a string built from at least three of its
attributes, and print it for one book.


In [16]:
# your code here


## Common errors

### TypeError: the class takes no arguments

A class with no `__init__` accepts no arguments, because there is no parameter list for them to go
into.


In [17]:
class Bare:
    pass


Bare("Tromso", [-4.1])


TypeError: Bare() takes no arguments

`Bare() takes no arguments` is the message. It is describing the call, not the class definition, so
the fix is to write the `__init__` the call assumes exists.

### TypeError: the count is one higher than you passed

Leaving `self` out of the `def` produces an error that looks wrong until you know why.


In [18]:
class NoSelf:
    def __init__(name, readings):
        self.name = name


NoSelf("Tromso", [-4.1])


TypeError: NoSelf.__init__() takes 2 positional arguments but 3 were given

Two arguments were passed and the message says three were given.

Python added the object as the first argument, as it always does. `__init__` declared two
parameters, `name` and `readings`, so the object landed in `name`, the string landed in `readings`,
and there was nowhere for the list to go.

Whenever an argument count in an error message is one higher than what you typed, this is why.

### AttributeError: `__init__` assigned to locals

Leaving `self.` off the left side is valid Python. It assigns a local variable to itself and then
throws it away.


In [19]:
class Local:
    def __init__(self, name, celsius):
        name = name
        celsius = celsius


reading = Local("Tromso", -4.1)
print("what it stored:", vars(reading))

reading.name


what it stored: {}


AttributeError: 'Local' object has no attribute 'name'

Creating the object raised nothing. `vars` shows the object is empty, which is the quickest
confirmation, and the error arrives later at whatever first reads an attribute.

### TypeError: `__init__` returned something

`__init__` fills in an object that already exists. Python returns the object, so `__init__` must
return `None`, which a function with no `return` does anyway.


In [20]:
class Returns:
    def __init__(self, name):
        self.name = name
        return self


Returns("Tromso")


TypeError: __init__() should return None, not 'Returns'

`__init__() should return None, not 'Returns'` is unusually direct about the rule.

This one catches people who expect `__init__` to work like a function that builds and hands back a
value. It does not. It is handed an object and fills it in.

### The quiet one: a mutable default is created once

The **Functions** notebook showed that a default argument is created once, when the function is
defined, and not on each call. `__init__` is a function, so the same applies, and the consequence
is that every object made without that argument shares one list.


In [21]:
class Shared:
    def __init__(self, name, readings=[]):
        self.name = name
        self.readings = readings


first = Shared("Tromso")
second = Shared("Bodo")
first.readings.append(-4.1)

print("first.readings: ", first.readings)
print("second.readings:", second.readings)
print("one list:       ", first.readings is second.readings)


first.readings:  [-4.1]
second.readings: [-4.1]
one list:        True


A reading was appended to one station and appeared on the other. Nothing raised, and both objects
report a value that is wrong for at least one of them.

The default `[]` was built once, when `class Shared` was executed, and every object without an
explicit `readings` got that same list. The fix is the one from **Functions**: default to `None`
and build the list inside.


In [22]:
class Fixed:
    def __init__(self, name, readings=None):
        self.name = name
        self.readings = [] if readings is None else readings


first = Fixed("Tromso")
second = Fixed("Bodo")
first.readings.append(-4.1)

print("first.readings: ", first.readings)
print("second.readings:", second.readings)
print("one list:       ", first.readings is second.readings)


first.readings:  [-4.1]
second.readings: []
one list:        False


Now each object gets a new list, because `[]` runs on every call rather than once at definition.

This matters more in a class than in a plain function, because objects are kept. A shared default
in a function usually shows up on the second call; a shared default in `__init__` can sit in a
program for months and surface as data appearing on a record nobody touched.


## Recap

- `class Name:` creates a type. Calling it, `Name(...)`, creates an object of that type.
- Creating an object is two steps: Python makes an empty one, then calls `__init__` on it.
- `self` is that object, passed in as the first argument. Python supplies it, so callers do not.
- `self` is an ordinary parameter name, not a keyword, and every Python programmer uses it.
- `__init__` is called for you. A method named anything else is not.
- `self.name = name` copies an argument, which is local to the call, onto the object, which is not.
- `__init__` fills in an object rather than building one, so it must not return anything.
- Every object carries its own attributes, and `vars(object)` shows them.
- Defaults make a parameter optional, not the attribute.
- Attributes can be assigned from outside, which is why two objects of one class can differ in
  what they carry.
- A mutable default in `__init__` is created once and shared by every object that relies on it.


## What is next

The **Methods** notebook. This one defined a single method, `average`, and used it without
explaining how the call works. That one covers methods properly: what `north.average()` does that
`Station.average()` cannot, methods that change the object against methods that return a value, and
how methods call one another.


---

&#8592; **Previous:** [Why Classes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/01-why-classes.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
